# Direct Two-Sum Legendre Wavelet Collocation

Legendre wavelet collocation benchmark for the singularly perturbed
boundary-value problem with Pe = 1, 10, 100, and 1000.

Includes the Pe = 1000 refinement study and coarse-versus-dense
evaluation-grid comparison.

In [ ]:
import os
import sys
import time
import platform
import subprocess
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from numpy.polynomial.legendre import Legendre, leggauss


# ============================================================
# 1. Settings
# ============================================================

PE_CASES = [
    {"Pe": 1.0, "J": 4, "M": 6},
    {"Pe": 10.0, "J": 8, "M": 6},
    {"Pe": 100.0, "J": 40, "M": 6},
    {"Pe": 1000.0, "J": 200, "M": 6},
]

REFINEMENT_PE = 1000.0
REFINEMENT_M = 6
REFINEMENT_J = [100, 150, 200, 250]
COARSE_POINTS = 101

U_LEFT = 1.0
U_RIGHT = 0.0
MAKE_PLOTS = True
SAVE_RESULTS = True
OUTPUT_DIR = "lwm_results"


# ============================================================
# 2. Exact solution
# ============================================================

def exact_solution(x, Pe, u_left=1.0, u_right=0.0):
    x = np.asarray(x, dtype=np.float64)
    if abs(Pe) < 1.0e-14:
        base = 1.0 - x
    else:
        base = (1.0 - np.exp(Pe * (x - 1.0))) / (1.0 - np.exp(-Pe))
    return u_right + (u_left - u_right) * base


# ============================================================
# 3. Direct two-sum LWM solver
# ============================================================

def solve_lwm_bvp(Pe, J, M, u_left=1.0, u_right=0.0):
    if J < 1 or M < 3:
        raise ValueError("Use J >= 1 and M >= 3.")

    LEG = [Legendre.basis(m) for m in range(M)]
    DLEG = [p.deriv(1) for p in LEG]
    DDLEG = [p.deriv(2) for p in LEG]

    # 3.1. Cell coordinates and basis functions

    def cell_bounds(j):
        return j / J, (j + 1) / J

    def local_coordinate(x, j):
        x_left, x_right = cell_bounds(j)
        h = x_right - x_left
        return 2.0 * (x - x_left) / h - 1.0

    def active_cell(x):
        if x <= 0.0:
            return 0
        if x >= 1.0:
            return J - 1
        return min(int(np.floor(x * J)), J - 1)

    def psi_on_cell(j, m, x):
        x_left, x_right = cell_bounds(j)
        h = x_right - x_left
        z = local_coordinate(x, j)
        scale = np.sqrt((2 * m + 1) / h)
        return scale * LEG[m](z)

    def dpsi_on_cell(j, m, x):
        x_left, x_right = cell_bounds(j)
        h = x_right - x_left
        z = local_coordinate(x, j)
        scale = np.sqrt((2 * m + 1) / h)
        return scale * (2.0 / h) * DLEG[m](z)

    def ddpsi_on_cell(j, m, x):
        x_left, x_right = cell_bounds(j)
        h = x_right - x_left
        z = local_coordinate(x, j)
        scale = np.sqrt((2 * m + 1) / h)
        return scale * (2.0 / h) ** 2 * DDLEG[m](z)

    # 3.2. Residual equations: J(M-2) rows

    assembly_start = time.perf_counter()
    z_nodes, _ = leggauss(M - 2)
    residual_points = []
    A_rows = []
    b_vals = []

    for j in range(J):
        x_left, x_right = cell_bounds(j)
        mid = 0.5 * (x_left + x_right)
        half = 0.5 * (x_right - x_left)

        for z in z_nodes:
            x = float(mid + half * z)
            residual_points.append(x)
            row = np.zeros(J * M, dtype=np.float64)
            for m in range(M):
                row[j * M + m] = ddpsi_on_cell(j, m, x) - Pe * dpsi_on_cell(j, m, x)
            A_rows.append(row)
            b_vals.append(0.0)

    # 3.3. Boundary conditions: 2 rows

    left_row = np.zeros(J * M, dtype=np.float64)
    right_row = np.zeros(J * M, dtype=np.float64)
    for m in range(M):
        left_row[m] = psi_on_cell(0, m, 0.0)
        right_row[(J - 1) * M + m] = psi_on_cell(J - 1, m, 1.0)
    A_rows.extend([left_row, right_row])
    b_vals.extend([u_left, u_right])

    # 3.4. Continuity of u and u': 2(J-1) rows

    for s in range(1, J):
        x_s = s / J
        left_cell, right_cell = s - 1, s
        value_row = np.zeros(J * M, dtype=np.float64)
        derivative_row = np.zeros(J * M, dtype=np.float64)

        for m in range(M):
            value_row[left_cell * M + m] = psi_on_cell(left_cell, m, x_s)
            value_row[right_cell * M + m] = -psi_on_cell(right_cell, m, x_s)
            derivative_row[left_cell * M + m] = dpsi_on_cell(left_cell, m, x_s)
            derivative_row[right_cell * M + m] = -dpsi_on_cell(right_cell, m, x_s)
        A_rows.extend([value_row, derivative_row])
        b_vals.extend([0.0, 0.0])

    A = np.vstack(A_rows)
    b = np.array(b_vals, dtype=np.float64)
    if A.shape != (J * M, J * M):
        raise ValueError("The system must contain exactly J*M equations.")
    assembly_time = time.perf_counter() - assembly_start

    # 3.5. Solve Ac = b and reconstruct u

    solve_start = time.perf_counter()
    coeff_vector = np.linalg.solve(A, b)
    solve_time = time.perf_counter() - solve_start
    coeffs = coeff_vector.reshape(J, M)
    algebraic_residual = np.linalg.norm(A @ coeff_vector - b, ord=np.inf)

    condition_start = time.perf_counter()
    condition_number = np.linalg.cond(A)
    condition_time = time.perf_counter() - condition_start

    def u_lwm(x):
        j = active_cell(x)
        total = 0.0
        for m in range(M):
            total += coeffs[j, m] * psi_on_cell(j, m, x)
        return total

    return {
        "Pe": Pe, "J": J, "M": M,
        "u_left": u_left, "u_right": u_right,
        "A": A, "b": b,
        "coeff_vector": coeff_vector, "coeffs": coeffs,
        "condition_number": condition_number,
        "algebraic_residual": algebraic_residual,
        "assembly_time": assembly_time, "solve_time": solve_time,
        "total_solver_time": assembly_time + solve_time,
        "condition_time": condition_time,
        "u_lwm": u_lwm,
        "residual_points": np.array(residual_points, dtype=np.float64),
    }


# ============================================================
# 4. Evaluation grids and error metrics
# ============================================================

def make_dense_grid():
    uniform_part = np.linspace(0.0, 1.0, 2001)
    layer_distance = np.geomspace(1.0e-14, 1.0, 5000)
    layer_part = 1.0 - layer_distance
    return np.unique(np.concatenate([uniform_part, layer_part, [0.0, 1.0]]))


def make_table_grid(Pe):
    standard = np.linspace(0.0, 1.0, 11)
    distances = np.array([5.0, 2.0, 1.0, 0.5, 0.2, 0.1, 0.05, 0.01, 0.0])
    layer_points = 1.0 - distances / Pe
    layer_points = layer_points[(layer_points >= 0.0) & (layer_points <= 1.0)]
    return np.unique(np.concatenate([standard, layer_points]))


def compute_error_metrics(x_grid, u_num, u_exact):
    abs_error = np.abs(u_num - u_exact)
    return {
        "abs_error": abs_error,
        "max_error": np.max(abs_error),
        "mean_error": np.mean(abs_error),
        "rms_error": np.sqrt(np.mean(abs_error ** 2)),
        "l2_error": np.sqrt(np.trapezoid(abs_error ** 2, x_grid)),
    }


def evaluate_solution(sol):
    x_dense = make_dense_grid()
    u_dense = np.array([sol["u_lwm"](float(x)) for x in x_dense], dtype=np.float64)
    u_exact = exact_solution(x_dense, sol["Pe"], sol["u_left"], sol["u_right"])
    info = compute_error_metrics(x_dense, u_dense, u_exact)
    info["x_dense"] = x_dense
    info["u_dense"] = u_dense
    info["u_exact_dense"] = u_exact
    info["left_boundary_error"] = abs(sol["u_lwm"](0.0) - sol["u_left"])
    info["right_boundary_error"] = abs(sol["u_lwm"](1.0) - sol["u_right"])
    return info


# ============================================================
# 5. Saving and plotting
# ============================================================

def save_csv(filename, rows, header):
    if SAVE_RESULTS:
        np.savetxt(os.path.join(OUTPUT_DIR, filename), rows,
                   delimiter=",", header=header, comments="")


def finish_plot(filename):
    plt.tight_layout()
    if SAVE_RESULTS:
        plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


def plot_solution(Pe, info):
    x = info["x_dense"]
    plt.figure(figsize=(7, 4.5))
    plt.plot(x, info["u_dense"], label="LWM")
    plt.plot(x, info["u_exact_dense"], "--", label="Exact")
    plt.xlabel("x")
    plt.ylabel("u(x)")
    plt.title(f"Direct Two-Sum LWM vs Exact, Pe={Pe:g}")
    plt.legend()
    finish_plot(f"lwm_pe{int(Pe)}_solution.png")

    plt.figure(figsize=(7, 4.5))
    plt.semilogy(x, info["abs_error"] + 1.0e-18)
    plt.xlabel("x")
    plt.ylabel("absolute error")
    plt.title(f"Absolute Error, Pe={Pe:g}")
    finish_plot(f"lwm_pe{int(Pe)}_error.png")

    if Pe in [100.0, 1000.0]:
        xi = Pe * (1.0 - x)
        mask = xi <= 10.0
        xi_layer = xi[mask]
        order = np.argsort(xi_layer)
        plt.figure(figsize=(7, 4.5))
        plt.plot(xi_layer[order], info["u_dense"][mask][order], label="LWM")
        plt.plot(xi_layer[order], info["u_exact_dense"][mask][order], "--", label="Exact")
        plt.xlabel(r"$\xi = Pe(1-x)$")
        plt.ylabel("u")
        plt.title(f"Boundary Layer in Stretched Coordinate, Pe={Pe:g}")
        plt.legend()
        finish_plot(f"lwm_stretched_pe{int(Pe)}.png")


# ============================================================
# 6. Main benchmark cases
# ============================================================

def run_main_cases():
    main_results = {}
    summary_rows = []

    for case in PE_CASES:
        Pe, J, M = float(case["Pe"]), int(case["J"]), int(case["M"])
        sol = solve_lwm_bvp(Pe, J, M, U_LEFT, U_RIGHT)
        info = evaluate_solution(sol)
        main_results[Pe] = {"sol": sol, "eval": info}
        tag = f"pe{int(Pe)}"

        print(f"\nLWM: Pe={Pe:g}, nu={1.0 / Pe:g}, J={J}, M={M}, unknowns={J * M}")
        print(f"Condition number cond(A)      = {sol['condition_number']:.6e}")
        print(f"Algebraic residual ||Ac-b||∞  = {sol['algebraic_residual']:.6e}")
        print(f"Assembly time [s]             = {sol['assembly_time']:.6f}")
        print(f"Linear solve time [s]         = {sol['solve_time']:.6f}")
        print(f"Total LWM solver time [s]     = {sol['total_solver_time']:.6f}")
        print(f"Maximum absolute error        = {info['max_error']:.6e}")
        print(f"Mean absolute error           = {info['mean_error']:.6e}")
        print(f"RMS error                     = {info['rms_error']:.6e}")
        print(f"L2-type error                 = {info['l2_error']:.6e}")
        print(f"Left boundary error           = {info['left_boundary_error']:.6e}")
        print(f"Right boundary error          = {info['right_boundary_error']:.6e}")

        table_rows = []
        print("\nx              LWM solution          Exact solution        Absolute error")
        for x in make_table_grid(Pe):
            numerical = sol["u_lwm"](float(x))
            exact = float(exact_solution(x, Pe, U_LEFT, U_RIGHT))
            error = abs(numerical - exact)
            table_rows.append([x, numerical, exact, error])
            print(f"{x: .10f}     {numerical: .12e}     {exact: .12e}     {error: .4e}")

        dense_rows = np.column_stack([
            info["x_dense"], info["u_dense"], info["u_exact_dense"], info["abs_error"]
        ])
        save_csv(f"lwm_{tag}_dense.csv", dense_rows, "x,u_lwm,u_exact,absolute_error")
        save_csv(f"lwm_{tag}_table.csv", table_rows, "x,u_lwm,u_exact,absolute_error")
        save_csv(f"lwm_{tag}_coefficients.csv", sol["coeffs"],
                 "Rows are cells j, columns are local Legendre modes m")

        if MAKE_PLOTS:
            plot_solution(Pe, info)

        summary_rows.append([
            Pe, 1.0 / Pe, J, M, J * M, sol["condition_number"],
            sol["algebraic_residual"], info["max_error"], info["mean_error"],
            info["rms_error"], info["l2_error"], info["left_boundary_error"],
            info["right_boundary_error"], sol["assembly_time"], sol["solve_time"],
            sol["total_solver_time"]
        ])

    print("\nFinal LWM Summary")
    print("Pe        J      M   unknowns      E_inf         E2        time [s]")
    for row in summary_rows:
        print(f"{row[0]:7.1f}  {int(row[2]):5d}  {int(row[3]):5d}  {int(row[4]):8d}  "
              f"{row[7]:.6e}  {row[10]:.6e}  {row[15]:.6f}")

    save_csv("lwm_multiple_pe_summary.csv", summary_rows,
             "Pe,nu,J,M,unknowns,condition_number,algebraic_residual_inf,"
             "max_absolute_error,mean_absolute_error,rms_error,l2_error,"
             "left_boundary_error,right_boundary_error,assembly_time,solve_time,"
             "total_solver_time")
    return main_results


# ============================================================
# 7. Refinement and coarse-versus-dense comparison
# ============================================================

def run_refinement(main_results):
    print(f"\nLWM Refinement Study: Pe={REFINEMENT_PE:g}, M={REFINEMENT_M}")
    rows = []
    existing = main_results.get(REFINEMENT_PE)

    for J in REFINEMENT_J:
        if (existing is not None and existing["sol"]["J"] == J
                and existing["sol"]["M"] == REFINEMENT_M):
            sol, info = existing["sol"], existing["eval"]
        else:
            sol = solve_lwm_bvp(REFINEMENT_PE, J, REFINEMENT_M, U_LEFT, U_RIGHT)
            info = evaluate_solution(sol)

        rows.append([
            REFINEMENT_PE, J, REFINEMENT_M, J * REFINEMENT_M,
            sol["condition_number"], sol["algebraic_residual"],
            info["max_error"], info["l2_error"], sol["assembly_time"],
            sol["solve_time"], sol["total_solver_time"]
        ])
        print(f"J={J:3d} | unknowns={J * REFINEMENT_M:4d} | "
              f"E_inf={info['max_error']:.6e} | E2={info['l2_error']:.6e} | "
              f"time={sol['total_solver_time']:.6f} s")

    save_csv(f"lwm_pe{int(REFINEMENT_PE)}_refinement.csv", rows,
             "Pe,J,M,unknowns,condition_number,algebraic_residual_inf,E_inf,E2,"
             "assembly_time,solve_time,total_solver_time")


def compare_coarse_dense(main_results):
    sol = main_results[1000.0]["sol"]
    info = main_results[1000.0]["eval"]
    x_coarse = np.linspace(0.0, 1.0, COARSE_POINTS)
    u_coarse = np.array([sol["u_lwm"](float(x)) for x in x_coarse], dtype=np.float64)
    u_exact = exact_solution(x_coarse, sol["Pe"], sol["u_left"], sol["u_right"])
    coarse_error = np.max(np.abs(u_coarse - u_exact))
    dense_error = info["max_error"]

    print(f"\nCoarse-versus-Dense Evaluation: Pe=1000, J={sol['J']}, M={sol['M']}")
    print(f"Coarse uniform grid: {COARSE_POINTS} points, E_inf={coarse_error:.6e}")
    print(f"Dense grid: {len(info['x_dense'])} points, E_inf={dense_error:.6e}")
    rows = [[sol["Pe"], sol["J"], sol["M"], COARSE_POINTS, coarse_error,
             len(info["x_dense"]), dense_error]]
    save_csv("lwm_pe1000_coarse_vs_dense.csv", rows,
             "Pe,J,M,coarse_points,coarse_E_inf,dense_points,dense_E_inf")


# ============================================================
# 8. Reproducibility information
# ============================================================

def cpu_model():
    try:
        if platform.system() == "Linux":
            with open("/proc/cpuinfo", encoding="utf-8") as file:
                for line in file:
                    if line.lower().startswith("model name"):
                        return line.split(":", 1)[1].strip()
        if platform.system() == "Darwin":
            return subprocess.check_output(
                ["sysctl", "-n", "machdep.cpu.brand_string"], text=True
            ).strip()
    except (OSError, subprocess.SubprocessError):
        pass
    name = platform.processor()
    if name.lower() in ["", "x86_64", "amd64", "arm64", "aarch64"]:
        return "Unavailable; record the CPU model manually."
    return name


def save_run_information():
    lines = [
        f"Python version: {sys.version.split()[0]}",
        f"NumPy version: {np.__version__}",
        f"Matplotlib version: {matplotlib.__version__}",
        f"Platform: {platform.platform()}",
        f"Machine: {platform.machine()}",
        f"Processor: {cpu_model()}",
        f"Logical CPU count: {os.cpu_count()}",
        "Arithmetic precision: float64",
        f"Main cases: {PE_CASES}",
        f"Refinement: Pe={REFINEMENT_PE}, M={REFINEMENT_M}, J={REFINEMENT_J}",
        f"Boundary values: u(0)={U_LEFT}, u(1)={U_RIGHT}",
        "Dense grid: sorted unique union of np.linspace(0, 1, 2001),",
        "1 - np.geomspace(1e-14, 1, 5000), and endpoints [0, 1].",
        f"Dense grid unique points: {len(make_dense_grid())}",
        f"Coarse grid: {COARSE_POINTS} uniformly spaced points including endpoints",
        "E_inf: maximum absolute error over the evaluation grid",
        "E2: square root of the trapezoidal integral of squared error",
        "Mean/RMS: unweighted sample statistics on the nonuniform dense grid",
        "LWM total solver time = assembly time + linear solve time",
        "Basis setup, diagnostics, evaluation, plotting and saving are excluded.",
    ]
    if SAVE_RESULTS:
        with open(os.path.join(OUTPUT_DIR, "lwm_run_information.txt"), "w",
                  encoding="utf-8") as file:
            file.write("\n".join(lines) + "\n")


# ============================================================
# 9. Main program
# ============================================================

def main():
    if SAVE_RESULTS:
        os.makedirs(OUTPUT_DIR, exist_ok=True)

    main_results = run_main_cases()
    run_refinement(main_results)
    compare_coarse_dense(main_results)
    save_run_information()

    if SAVE_RESULTS:
        print(f"\nSaved files inside folder: {OUTPUT_DIR}/")
    print("\nLWM BENCHMARK COMPLETE")


if __name__ == "__main__":
    main()
